# Análisis temporal — comparabilidad y replicación por periodo (R2-8)

Notebook independiente para responder al Revisor 2 (comentario 8): documenta comparabilidad de escalas entre las 4 convocatorias (2021-2 a 2023-1), reporta composición de clústeres por periodo, y reajusta el pipeline POR SEPARADO en cada periodo (replicación temporal).

**Cómo correrlo:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno` → CPU (no necesita GPU).
2. Ajusta la ruta de tu `df_maestra.csv` en la celda de carga de datos si no está en `/content/drive/MyDrive/Proyecto/`.
3. Corre todas las celdas en orden (`Entorno de ejecución` → `Ejecutar todas`).
4. Los resultados se guardan automáticamente en tu Drive, en `/content/drive/MyDrive/Proyecto/analisis_temporal/`, por si la sesión se desconecta.

In [ ]:
!pip install -q umap-learn

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
import umap.umap_ as umap_cpu
import warnings
warnings.filterwarnings("ignore")
print("✅ Librerías listas")

## 1. Cargar los datos desde Google Drive

Ajusta la ruta si tu `df_maestra.csv` está en otra carpeta de tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Proyecto/df_maestra.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^Column1')]
OUT_DIR = '/content/drive/MyDrive/Proyecto/analisis_temporal'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Shape original: {df.shape}")
df.head(3)

## 2. Preprocesamiento (idéntico al notebook original)

In [ ]:
def preprocesar_saber_pro(df_raw, sample_n=200_000, random_state=42):
    df_limpio = df_raw.copy()

    mapa_bano = {'1': 1, '2': 2, '3 o 4': 3, '5 o 6': 5, 'MAS DE 6': 6, 'NINGUNA': 0}
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                     'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 millón': 2,
        'Entre 1 millón y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'Más de 7 millones': 7}
    mapa_educ = {
        'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
        'Secundaria (Bachillerato) incompleta': 3,
        'Secundaria (Bachillerato) completa': 4,
        'Técnica o tecnológica incompleta': 5,
        'Técnica o tecnológica completa': 6,
        'Educación profesional incompleta': 7,
        'EDUCACIÓN PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                    'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}
    mapeo_semestre = {str(i).zfill(2): i for i in range(1, 12)}
    mapeo_semestre['12 o más'] = 12

    mapeables = {
        'FAMI_CUANTOSCOMPARTEBAÑO':      mapa_bano,
        'FAMI_ESTRATOVIVIENDA':          mapa_estrato,
        'ESTU_VALORMATRICULAUNIVERSIDAD': mapa_valormatricula,
        'FAMI_EDUCACIONPADRE':           mapa_educ,
        'FAMI_EDUCACIONMADRE':           mapa_educ,
        'ESTU_HORASSEMANATRABAJA':       mapeo_horas,
        'ESTU_SEMESTRECURSA':            mapeo_semestre,
    }
    for col, mapa in mapeables.items():
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].map(mapa)

    columnas_puntaje = [
        'MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
        'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
    columnas_ordinales = [
        'FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
        'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
    columnas_nominales = [
        'ESTU_TITULOOBTENIDOBACHILLER',
        'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
        'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
        'ESTU_COMOCAPACITOEXAMENSB11',
        'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
        'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA']
    col_geo = 'ESTU_COD_DEPTO_PRESENTACION'

    for col in columnas_puntaje:
        if col in df_limpio.columns:
            df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
    for col in columnas_ordinales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())
    for col in columnas_nominales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode(dropna=True)[0])

    cols_usar = columnas_ordinales + columnas_puntaje + columnas_nominales
    cols_extra = [col_geo, 'INST_COD_INSTITUCION', 'ESTU_PRGM_ACADEMICO',
                  'PERIODO', 'PUNT_GLOBAL', 'ESTU_CONSECUTIVO']
    cols_df = cols_usar + [c for c in cols_extra if c in df_limpio.columns]
    df_filtrado = df_limpio[[c for c in cols_df if c in df_limpio.columns]].copy()
    df_filtrado = df_filtrado.dropna(subset=[c for c in columnas_puntaje if c in df_filtrado.columns])
    print(f"Filas después de limpieza: {len(df_filtrado):,}")

    cols_punt = [c for c in columnas_puntaje if c in df_filtrado.columns]
    cols_ord = [c for c in columnas_ordinales if c in df_filtrado.columns]
    cols_nom = [c for c in columnas_nominales if c in df_filtrado.columns]

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_punt = scaler.fit_transform(df_filtrado[cols_punt])
    X_ohe = encoder.fit_transform(df_filtrado[cols_nom])
    X = np.hstack([df_filtrado[cols_ord].values, X_punt, X_ohe])
    feature_names = (cols_ord + list(scaler.get_feature_names_out(cols_punt))
                      + list(encoder.get_feature_names_out(cols_nom)))

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy='median').fit_transform(X)
        print("NaN residuales imputados con mediana")

    if sample_n is not None and sample_n < df_filtrado.shape[0]:
        rng = np.random.default_rng(seed=random_state)
        idx = rng.choice(df_filtrado.shape[0], size=sample_n, replace=False)
        df_filtrado = df_filtrado.iloc[idx].reset_index(drop=True)
        X = X[idx]

    print(f"Preprocesamiento completo — shape X: {X.shape}")
    return df_limpio, df_filtrado, X, feature_names, encoder, scaler


In [ ]:
df_limpio_full, df_filtrado_full, X_full, feature_names, encoder, scaler = \
    preprocesar_saber_pro(df, sample_n=None)
n_total = X_full.shape[0]
print(f"Shape X_full: {X_full.shape}")

## 3. (a) Comparabilidad de puntajes entre periodos

In [ ]:
periodos = df_filtrado_full["PERIODO"].values
cols_punt_temporal = ['MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
                      'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT',
                      'MOD_COMUNI_ESCRITA_PUNT', 'PUNT_GLOBAL']
for col in cols_punt_temporal:
    stats = df_filtrado_full.groupby('PERIODO')[col].agg(['mean', 'std', 'count'])
    print(f"\n{col}:")
    print(stats)

## 4. (b) Partición 'publicada' de referencia y su composición por periodo

In [ ]:
# Partición base K=8 (misma metodología del manuscrito: UMAP fit sobre
# 80,000 filas, transform sobre el resto; K-Means K=8), para tener una
# referencia 'publicada' propia sin depender de otros notebooks.
SEED_BASE = 42
K = 8
N_FIT = 80_000
N_EVAL = 50_000

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

rng_fit = np.random.default_rng(seed=SEED_BASE)
idx_fit = rng_fit.choice(n_total, size=N_FIT, replace=False)
rng_eval = np.random.default_rng(seed=0)
idx_eval = rng_eval.choice(n_total, size=N_EVAL, replace=False)

log("Ajustando UMAP base (K=8, seed=42)...")
reducer_base = umap_cpu.UMAP(n_components=2, random_state=SEED_BASE, n_neighbors=10,
                              low_memory=True, n_jobs=-1)
reducer_base.fit(X_full[idx_fit])
emb_eval_base = reducer_base.transform(X_full[idx_eval])
km_base = MiniBatchKMeans(n_clusters=K, random_state=SEED_BASE, n_init="auto", batch_size=10_000)
labels_pub = km_base.fit_predict(emb_eval_base)
log(f"Partición base lista. Tamaños: {np.bincount(labels_pub)}")

In [ ]:
periodos_eval = periodos[idx_eval]
df_comp = pd.DataFrame({'cluster': labels_pub, 'periodo': periodos_eval})
tabla_comp = pd.crosstab(df_comp['periodo'], df_comp['cluster'], normalize='index') * 100
print(tabla_comp.round(1))
tabla_comp.to_csv(os.path.join(OUT_DIR, 'composicion_cluster_por_periodo.csv'))

## 5. (c) Pipeline ajustado POR SEPARADO en cada periodo
⏱️ Ajusta UMAP+K-Means 4 veces (una por periodo) — puede tardar 15-25 minutos.

In [ ]:
from scipy.optimize import linear_sum_assignment

resultados_periodo = {}
centroides_por_periodo = {}
tam_por_periodo = {}

periodos_lista = sorted(pd.unique(periodos))
for p in periodos_lista:
    t_p = time.time()
    idx_p = np.where(periodos == p)[0]
    X_p = X_full[idx_p]
    n_p = X_p.shape[0]
    print(f"[{p}] n={n_p:,}  ajustando UMAP...")
    seed_p = 700 + periodos_lista.index(p)
    reducer_p = umap_cpu.UMAP(n_components=2, random_state=seed_p, n_neighbors=10,
                               low_memory=True, n_jobs=-1)
    if n_p > 100_000:
        rng = np.random.default_rng(seed_p)
        idx_fit_p = rng.choice(n_p, size=80_000, replace=False)
        reducer_p.fit(X_p[idx_fit_p])
        emb_p = reducer_p.transform(X_p)
    else:
        emb_p = reducer_p.fit_transform(X_p)
    km_p = MiniBatchKMeans(n_clusters=K, random_state=seed_p, n_init="auto", batch_size=10_000)
    labels_p = km_p.fit_predict(emb_p)
    tam_por_periodo[str(p)] = (np.bincount(labels_p, minlength=K) / n_p * 100).tolist()
    centroides = np.array([X_p[labels_p == c].mean(axis=0) if (labels_p == c).sum() > 0
                            else np.full(X_p.shape[1], np.nan) for c in range(K)])
    centroides_por_periodo[str(p)] = centroides
    print(f"[{p}] listo en {time.time()-t_p:.1f}s. Tamaños (%): {np.round(tam_por_periodo[str(p)], 1)}")

## 6. Emparejar clústeres entre periodos (distancia de centroides) y correlación

In [ ]:
ref_p = str(periodos_lista[0])
ref_cent = centroides_por_periodo[ref_p]
emparejamientos = {}
for p in periodos_lista[1:]:
    p = str(p)
    cent_p = centroides_por_periodo[p]
    cost = np.linalg.norm(ref_cent[:, None, :] - cent_p[None, :, :], axis=2)
    row_ind, col_ind = linear_sum_assignment(cost)
    corrs = [float(np.corrcoef(ref_cent[a], cent_p[b])[0, 1]) for a, b in zip(row_ind, col_ind)]
    emparejamientos[p] = {'correlacion_media': float(np.mean(corrs)),
                          'correlaciones_por_cluster': [round(c, 2) for c in corrs]}
    print(f"[{ref_p} vs {p}] correlación media de centroides emparejados = {np.mean(corrs):.3f}")

json.dump({'tamanos_por_periodo_pct': tam_por_periodo, 'emparejamientos_vs_referencia': emparejamientos},
          open(os.path.join(OUT_DIR, 'resultados.json'), 'w'), indent=2)
print("\nValores esperados (manuscrito): correlaciones medias 0.917-0.956 entre periodos.")

## 7. Figura (Supplementary Figure S7)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

periodos_comp = [str(p) for p in periodos_lista[1:]]
labels_leyenda = [f'2021-2 vs {p[:4]}-{p[4]}' for p in periodos_comp]
colors = ['#3B6FA0', '#4C9F70', '#B33F3F']

x = np.arange(K)
width = 0.26
fig, ax = plt.subplots(figsize=(11, 5.5), dpi=150)
for i, (p, lab, col) in enumerate(zip(periodos_comp, labels_leyenda, colors)):
    corrs = emparejamientos[p]['correlaciones_por_cluster']
    ax.bar(x + (i - 1) * width, corrs, width, label=lab, color=col)
ax.axhline(0.8, color='gray', linestyle='--', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([f'C{c}' for c in range(K)])
ax.set_xlabel('Cl\u00faster de referencia (2021-2)')
ax.set_ylabel('Correlaci\u00f3n del centroide emparejado')
ax.set_title('Estabilidad temporal de los perfiles:\ncorrelaci\u00f3n de centroides emparejados entre periodos (pipeline reajustado por separado)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figura_estabilidad_temporal.png'), dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("\u2705 Figura guardada en Drive (Supplementary Figure S7)")
